# Arm D: exact policy gradient + expanded data (Qwen2.5-0.5B, T4)

Closed-form expected-reward objective (the verifier prices both decisions,
so no sampling and zero estimator variance) + class-balanced rewards +
an expanded, leakage-guarded training corpus.

**Leakage discipline**: training uses `expanded_train.jsonl` (seed 101),
tuning/monitoring uses `expanded_val.jsonl` (seed 202), both deduplicated
against the committed `benchmark_test.jsonl` at the decision-context
level. The committed test set is evaluated ONCE, at the end.

Runtime → Change runtime type → **T4 GPU** → Save, then Run all (~2.5 h).

In [ ]:
!git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
%cd verifier-as-reward
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
# guardrail: the verifier (reward source) must be intact
!PYTHONPATH=. python test_authority_verifier.py

In [ ]:
# expanded train + val corpora, deduplicated against the committed test set
!PYTHONPATH=. python make_expanded_train.py \
  --train-seed 101 --train-traces-per-class 150 \
  --val-seed 202 --val-traces-per-class 25

In [ ]:
# Arm D: exact PG, 3 seeds. Training curve is monitored on the VAL set.
!for s in 7 8 9; do \
  PYTHONPATH=. python train_verifier_reward.py --model Qwen/Qwen2.5-0.5B \
    --exact-pg --balance-reward \
    --steps 400 --batch-size 16 --lr 1e-5 \
    --train-file expanded_train.jsonl --test-file expanded_val.jsonl \
    --eval-every 25 --eval-max-actions 120 \
    --seed $s --log-file training_log_qwen05b_exactpg_seed$s.jsonl \
    --save-dir ckpt_exactpg_seed$s  || break ; done

In [ ]:
# FINAL evaluation — the committed held-out test set, natural-language
# prompts + parsing identical to the API-model ladder. Run once.
import json
json.dump({"backends": {}}, open("results_exactpg.json", "w"))
!for s in 7 8 9; do \
  PYTHONPATH=. python train_verifier_reward.py \
    --eval-checkpoint ckpt_exactpg_seed$s --test-file benchmark_test.jsonl \
    --merge-results results_exactpg.json  || break ; done

In [ ]:
# Optional: keep the trained weights across sessions
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r ckpt_exactpg_seed7 /content/drive/MyDrive/
!zip -j exactpg_results.zip training_log_qwen05b_exactpg_seed*.jsonl results_exactpg.json
from google.colab import files
files.download("exactpg_results.zip")